# Landmine detection: CBAM study, evaluation notebook (E4)

1. **Inputs** (*Add Input* → *Your Work*): the finished versions of `01_train_kaggle` (sessions A, B, C, ...).
   Optionally also a dataset containing the 2025 **YOLO11m** `best.pt` (set `LEGACY_YOLO11M` below); in that
   case attach the MineGuard-Thermal dataset too, because those weights are evaluated on its test split.
2. **Settings:** Accelerator **GPU T4** (one is enough), Internet **on**. Click *Run All*.
3. Outputs: `results/summary.md` (mean ± std over seeds, **test** split), `cbam_vs_baseline.csv` (paired
   per-seed differences), `runs.csv`, `latency.json`, plus plots. Download `results/` and send it back.

In [ ]:
# ============================ CONFIGURATION ============================
INPUT_ROOTS = ["/kaggle/input"]            # where the training outputs are attached
LEGACY_YOLO11M = None                      # e.g. "/kaggle/input/fyp-weights/best.pt" (2025 YOLO11m, original split)
IMGSZ = 416
YOLOV5_COMMIT = "f9578796b8a64889aa9e66dc06914d19f40d338e"
ULTRALYTICS_VERSION = "8.4.175"
SMOKE_TEST = False
WORK_DIR, OUT_DIR = "/tmp/lm", "/kaggle/working/results"
# =======================================================================
SRC_DIR, YOLO_DIR, GEN_DIR = f"{WORK_DIR}/src", f"{WORK_DIR}/yolov5", f"{WORK_DIR}/gen"

In [ ]:
import os, sys, json, platform, subprocess, time
import torch

os.makedirs(SRC_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"  # YOLOv5/Ultralytics checkpoints are full pickles


def sh(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)


if not os.path.isdir(YOLO_DIR):
    sh(f"git clone -q https://github.com/ultralytics/yolov5 {YOLO_DIR}")
sh(f"git checkout -q {YOLOV5_COMMIT}", cwd=YOLO_DIR)
sh("pip install -q -r requirements.txt", cwd=YOLO_DIR)
sh(f"pip install -q ultralytics=={ULTRALYTICS_VERSION}")

GPUS = list(range(torch.cuda.device_count())) or ["cpu"]
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] or "none (CPU only!)")
if GPUS == ["cpu"] and not SMOKE_TEST:
    raise SystemExit("No GPU found: set Accelerator to 'GPU T4 x2' in the notebook settings.")

In [ ]:
%%writefile /tmp/lm/src/splits.py
"""Dataset discovery and split construction for MineGuard-Thermal v15 (YOLO format).

Three splits are built from the same 3,042 images:

* ``random`` - the original Roboflow train/valid/test split used in the thesis.
* ``scene``  - whole recording sequences are held out, so no test sequence is seen in training.
* ``*_smoke`` - tiny subsets of the above, used to check the pipeline end to end in minutes.

Split folders contain links to the original files, so building them is cheap.
"""

import csv
import os
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

EXPECTED_COUNTS = {"train": 2132, "valid": 607, "test": 303}
IMG_EXTS = {".jpg", ".jpeg", ".png"}
SPLITS = ("train", "valid", "test")

# Scene-disjoint split (experiment E3): these sequences are held out entirely.
# Every other sequence (illia_1/2, MYKHAILO_1/2, DM-11, FlightAltitude) goes to train.
SCENE_HOLDOUT = {
    "test": ("illia_4", "MYKHAILO_4"),
    "valid": ("illia_3", "MYKHAILO_3"),
}


def sequence_of(filename):
    """Map a Roboflow filename to the recording sequence it came from (None if unknown)."""
    stem = filename.split(".rf.")[0]
    if stem.startswith("FlightAltitude"):
        return "FlightAltitude"
    m = re.match(r"(MYKHAILO_\d+|illia_\d+|DM-11)_", stem)
    return m.group(1) if m else None


def find_dataset(search_roots):
    """Return the first directory under ``search_roots`` that holds a YOLO export with train/valid/test."""
    for root in search_roots:
        if not Path(root).exists():
            continue
        for dirpath, dirnames, _ in os.walk(root):
            if set(SPLITS) <= set(dirnames) and Path(dirpath, "train", "images").is_dir():
                return Path(dirpath)
    return None


def download_roboflow(dest, api_key):
    """Fallback when the dataset is not attached as a Kaggle input."""
    from roboflow import Roboflow

    project = Roboflow(api_key=api_key).workspace("dronecv").project("mineguard-thermal")
    ds = project.version(15).download("yolov5", location=str(dest), overwrite=False)
    return Path(ds.location)


def collect_items(src_root):
    """List (original_split, image_path, label_path_or_None, n_boxes) for every image."""
    items = []
    for sp in SPLITS:
        for img in sorted((src_root / sp / "images").iterdir()):
            if img.suffix.lower() not in IMG_EXTS:
                continue
            lab = src_root / sp / "labels" / (img.stem + ".txt")
            n = sum(1 for line in lab.read_text().splitlines() if line.strip()) if lab.exists() else 0
            items.append((sp, img, lab if lab.exists() else None, n))
    return items


def _link(src, dst):
    """Symlink if possible (Kaggle/Linux), otherwise copy (e.g. Windows without dev mode)."""
    if dst.exists():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def _write_split(assignment, out_dir):
    """Materialise {split: [items]} under ``out_dir`` and write a data.yaml for YOLOv5/Ultralytics."""
    if out_dir.exists():
        shutil.rmtree(out_dir)
    for sp in SPLITS:
        (out_dir / sp / "images").mkdir(parents=True, exist_ok=True)
        (out_dir / sp / "labels").mkdir(parents=True, exist_ok=True)
        for _, img, lab, _ in assignment[sp]:
            _link(img, out_dir / sp / "images" / img.name)
            if lab is not None:
                _link(lab, out_dir / sp / "labels" / lab.name)
    yaml_path = out_dir / "data.yaml"
    yaml_path.write_text(
        f"path: {out_dir.as_posix()}\n"
        "train: train/images\nval: valid/images\ntest: test/images\n"
        "nc: 1\nnames: ['landmine']\n"
    )
    return yaml_path


def _summarise(name, assignment):
    print(f"\n[{name}] images / boxes per split and sequence")
    seqs = sorted({sequence_of(i[1].name) or "?" for sp in SPLITS for i in assignment[sp]})
    print(f"  {'sequence':16s}" + "".join(f"{sp:>16s}" for sp in SPLITS))
    for seq in seqs:
        row = ""
        for sp in SPLITS:
            sel = [i for i in assignment[sp] if (sequence_of(i[1].name) or "?") == seq]
            row += f"{len(sel):>9d} / {sum(i[3] for i in sel):<5d}"
        print(f"  {seq:16s}{row}")
    print(f"  {'TOTAL':16s}" + "".join(
        f"{len(assignment[sp]):>9d} / {sum(i[3] for i in assignment[sp]):<5d}" for sp in SPLITS))


def build_splits(src_root, out_root, smoke=False, smoke_sizes=(64, 32, 32), seed=0, manifest_path=None):
    """Build the random and scene splits (and smoke subsets). Returns {split_name: data.yaml path}."""
    items = collect_items(src_root)
    counts = Counter(i[0] for i in items)
    if dict(counts) != EXPECTED_COUNTS:
        print(f"WARNING: expected {EXPECTED_COUNTS}, found {dict(counts)} - is this MineGuard-Thermal v15?")

    random_split = {sp: [i for i in items if i[0] == sp] for sp in SPLITS}

    holdout = {seq: sp for sp, seqs in SCENE_HOLDOUT.items() for seq in seqs}
    scene_split = defaultdict(list)
    unknown = []
    for it in items:
        seq = sequence_of(it[1].name)
        if seq is None:
            unknown.append(it[1].name)
        scene_split[holdout.get(seq, "train")].append(it)
    if unknown:
        print(f"WARNING: {len(unknown)} images with unknown sequence were put in train, e.g. {unknown[:3]}")
    leaked = {sequence_of(i[1].name) for i in scene_split["train"]} & set(holdout)
    assert not leaked, f"held-out sequences leaked into train: {leaked}"

    splits = {"random": random_split, "scene": dict(scene_split)}
    if smoke:
        rng = random.Random(seed)
        splits = {
            f"{name}_smoke": {sp: rng.sample(s[sp], min(n, len(s[sp]))) for sp, n in zip(SPLITS, smoke_sizes)}
            for name, s in splits.items()
        }

    yamls = {}
    for name, assignment in splits.items():
        _summarise(name, assignment)
        yamls[name] = _write_split(assignment, Path(out_root) / name)

    if manifest_path is not None:
        Path(manifest_path).parent.mkdir(parents=True, exist_ok=True)
        scene_of = {i[1].name: sp for sp in SPLITS for i in scene_split[sp]}
        with open(manifest_path, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["image", "sequence", "boxes", "random_split", "scene_split"])
            for sp, img, _, n in items:
                w.writerow([img.name, sequence_of(img.name) or "", n, sp, scene_of[img.name]])
    return yamls

In [ ]:
%%writefile /tmp/lm/src/setup_yolov5.py
"""Add CBAM to a pinned Ultralytics YOLOv5 checkout and prepare fair, COCO-pretrained CBAM models.

What this does (each step is idempotent):

1. ``patch``     - appends the CBAM module to ``models/common.py`` and teaches ``parse_model`` about it.
2. ``make_yaml`` - writes ``yolov5{s,x}_cbam.yaml``: the official architecture with one CBAM after
                   *every* C3 block in the backbone and the neck (8 CBAMs), all layer references remapped.
3. ``init_ckpt`` - remaps the official COCO ``yolov5{s,x}.pt`` weights into the CBAM layer numbering, so
                   the CBAM model starts from exactly the same pretrained weights as the baseline. Only
                   the CBAM blocks (and, as for the baseline, the 1-class Detect convs) start from scratch.
                   Without this remap YOLOv5 matches weights by layer index and silently drops most of them.
4. ``info``      - parameter / GFLOP table for the four models.

Run as a script inside any Python that can import the YOLOv5 repo, e.g.
``python setup_yolov5.py all --yolo-dir /tmp/lm/yolov5 --gen-dir /tmp/lm/gen --seeds 0 1 2``.
"""

import argparse
import copy
import json
import os
import sys
from pathlib import Path

import yaml

CBAM_SOURCE = '''

# ---------------------------------------------------------------------------------------------
# CBAM: Convolutional Block Attention Module (Woo et al., ECCV 2018), added for the landmine study.
# Pooling uses mean/amax rather than AdaptiveAvg/MaxPool2d because YOLOv5 trains with
# torch.use_deterministic_algorithms(True) and adaptive pooling has no deterministic CUDA backward.
class ChannelAttention(nn.Module):
    """Channel attention: shared MLP over global average- and max-pooled descriptors."""

    def __init__(self, c, r=16):
        super().__init__()
        h = max(c // r, 1)
        self.mlp = nn.Sequential(nn.Conv2d(c, h, 1, bias=False), nn.ReLU(), nn.Conv2d(h, c, 1, bias=False))

    def forward(self, x):
        return torch.sigmoid(self.mlp(x.mean((2, 3), keepdim=True)) + self.mlp(x.amax((2, 3), keepdim=True)))


class SpatialAttention(nn.Module):
    """Spatial attention: 7x7 conv over channel-wise average and max maps."""

    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k // 2, bias=False)

    def forward(self, x):
        return torch.sigmoid(self.conv(torch.cat((x.mean(1, keepdim=True), x.amax(1, keepdim=True)), 1)))


class CBAM(nn.Module):
    """Channel attention followed by spatial attention; output has the same shape as the input."""

    def __init__(self, c1, r=16, k=7):
        super().__init__()
        self.ca = ChannelAttention(c1, r)
        self.sa = SpatialAttention(k)

    def forward(self, x):
        x = x * self.ca(x)
        return x * self.sa(x)
'''

PARSE_ANCHOR = "        elif m is nn.BatchNorm2d:\n"
PARSE_BRANCH = "        elif m is CBAM:\n            c2 = ch[f]\n            args = [c2, *args]\n"
IMPORT_ANCHOR = "from models.experimental import MixConv2d\n"
IMPORT_LINE = "from models.common import CBAM\n"


def patch(yolo_dir):
    """Add CBAM to models/common.py and models/yolo.py (safe to call repeatedly)."""
    common = Path(yolo_dir, "models", "common.py")
    text = common.read_text(encoding="utf-8")
    if "class CBAM(" not in text:
        common.write_text(text + CBAM_SOURCE, encoding="utf-8")

    yolo = Path(yolo_dir, "models", "yolo.py")
    text = yolo.read_text(encoding="utf-8")
    if IMPORT_LINE not in text:
        assert text.count(IMPORT_ANCHOR) == 1 and text.count(PARSE_ANCHOR) == 1, (
            "models/yolo.py does not look like the pinned YOLOv5 commit; refusing to patch")
        text = text.replace(IMPORT_ANCHOR, IMPORT_LINE + IMPORT_ANCHOR, 1)
        text = text.replace(PARSE_ANCHOR, PARSE_BRANCH + PARSE_ANCHOR, 1)
        yolo.write_text(text, encoding="utf-8")
    print("patched YOLOv5 with CBAM")


def make_yaml(yolo_dir, variant, gen_dir):
    """Write yolov5{variant}_cbam.yaml; return (path, module_index_map, cbam_indices)."""
    src = yaml.safe_load(Path(yolo_dir, "models", f"yolov5{variant}.yaml").read_text(encoding="utf-8"))
    layers = src["backbone"] + src["head"]
    n_backbone = len(src["backbone"])

    new, module_at, output_at, cbam_idx = [], {}, {}, []
    new_backbone_len = None
    for i, layer in enumerate(layers):
        if i == n_backbone:
            new_backbone_len = len(new)
        module_at[i] = output_at[i] = len(new)
        new.append(copy.deepcopy(layer))
        if layer[2] == "C3":
            cbam_idx.append(len(new))
            output_at[i] = len(new)  # anything that consumed this C3 now consumes its CBAM
            new.append([-1, 1, "CBAM", []])

    def remap(f):
        if isinstance(f, list):
            return [remap(x) for x in f]
        return f if f < 0 else output_at[f]

    for layer in new:
        layer[0] = remap(layer[0])

    out = {k: v for k, v in src.items() if k not in ("backbone", "head")}
    out["backbone"], out["head"] = new[:new_backbone_len], new[new_backbone_len:]
    path = Path(gen_dir, f"yolov5{variant}_cbam.yaml")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(f"# yolov5{variant}.yaml with CBAM after every C3 (generated by setup_yolov5.py)\n"
                    + yaml.safe_dump(out, sort_keys=False, default_flow_style=None), encoding="utf-8")
    meta = {"module_map": module_at, "cbam_indices": cbam_idx}
    Path(gen_dir, f"yolov5{variant}_cbam.map.json").write_text(json.dumps(meta, indent=1))
    return path, module_at, cbam_idx


def _import_yolov5(yolo_dir):
    sys.path.insert(0, str(Path(yolo_dir).resolve()))
    os.chdir(yolo_dir)  # YOLOv5 resolves some paths relative to the repo root


def init_ckpt(yolo_dir, variant, seed, gen_dir):
    """Create gen_dir/yolov5{variant}_cbam_init_s{seed}.pt from the official COCO weights."""
    out_path = Path(gen_dir, f"yolov5{variant}_cbam_init_s{seed}.pt").resolve()
    if out_path.exists():
        return out_path
    cfg, module_at, cbam_idx = make_yaml(yolo_dir, variant, gen_dir)
    cfg = cfg.resolve()
    _import_yolov5(yolo_dir)
    import torch
    from models.yolo import Model
    from ultralytics.utils.patches import torch_load
    from utils.downloads import attempt_download

    ckpt = torch_load(attempt_download(f"yolov5{variant}.pt"), map_location="cpu")
    coco = ckpt["model"].float().state_dict()

    remapped = {}
    for k, v in coco.items():
        parts = k.split(".")
        parts[1] = str(module_at[int(parts[1])])
        remapped[".".join(parts)] = v

    torch.manual_seed(seed)  # CBAM blocks get a seed-dependent random init, like any new layer
    model = Model(str(cfg), ch=3, nc=80)
    missing, unexpected = model.load_state_dict(remapped, strict=False)
    assert not unexpected, f"unexpected keys: {unexpected[:5]}"
    bad = [k for k in missing if int(k.split(".")[1]) not in cbam_idx]
    assert not bad, f"non-CBAM weights missing after remap: {bad[:5]}"
    print(f"yolov5{variant}_cbam seed {seed}: {len(remapped)} COCO tensors transferred, "
          f"{len(missing)} CBAM tensors initialised from scratch")

    torch.save({"model": copy.deepcopy(model).half(), "epoch": -1, "best_fitness": None, "optimizer": None,
                "ema": None, "updates": None, "opt": None,
                "note": f"COCO yolov5{variant}.pt remapped into the CBAM layout (seed {seed})"}, out_path)
    return out_path


def info(yolo_dir, gen_dir, imgsz=416):
    """Print and return params/GFLOPs for the baseline and CBAM models (nc=1)."""
    _import_yolov5(yolo_dir)
    import thop
    import torch
    from models.common import CBAM
    from models.yolo import Model

    rows = {}
    for variant in ("s", "x"):
        for name, cfg in ((f"yolov5{variant}", Path(yolo_dir, "models", f"yolov5{variant}.yaml")),
                          (f"yolov5{variant}_cbam", Path(gen_dir, f"yolov5{variant}_cbam.yaml").resolve())):
            if not cfg.exists():
                continue
            m = Model(str(cfg), ch=3, nc=1).eval()
            params = sum(p.numel() for p in m.parameters())
            macs = thop.profile(copy.deepcopy(m), inputs=(torch.zeros(1, 3, imgsz, imgsz),), verbose=False)[0]
            n_cbam = sum(isinstance(x, CBAM) for x in m.modules())
            rows[name] = {"params": params, "gflops": 2 * macs / 1e9, "cbam_blocks": n_cbam}
    print(f"\n{'model':16s}{'params':>14s}{'GFLOPs@' + str(imgsz):>14s}{'CBAM blocks':>13s}")
    for k, r in rows.items():
        print(f"{k:16s}{r['params']:>14,d}{r['gflops']:>14.1f}{r['cbam_blocks']:>13d}")
    for v in ("s", "x"):
        if f"yolov5{v}_cbam" in rows:
            assert rows[f"yolov5{v}_cbam"]["cbam_blocks"] == 8, "expected one CBAM after each of the 8 C3 blocks"
    return rows


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("step", choices=["patch", "yaml", "init", "info", "all"])
    ap.add_argument("--yolo-dir", required=True)
    ap.add_argument("--gen-dir", required=True)
    ap.add_argument("--variants", nargs="+", default=["s", "x"])
    ap.add_argument("--seeds", nargs="+", type=int, default=[0])
    ap.add_argument("--imgsz", type=int, default=416)
    a = ap.parse_args()
    a.gen_dir = str(Path(a.gen_dir).resolve())
    a.yolo_dir = str(Path(a.yolo_dir).resolve())
    if a.step in ("patch", "all"):
        patch(a.yolo_dir)
    if a.step in ("yaml", "all"):
        for v in a.variants:
            print("wrote", make_yaml(a.yolo_dir, v, a.gen_dir)[0])
    if a.step in ("init", "all"):
        here = os.getcwd()
        for v in a.variants:
            for s in a.seeds:
                init_ckpt(a.yolo_dir, v, s, a.gen_dir)
                os.chdir(here)
    if a.step in ("info", "all"):
        res = info(a.yolo_dir, a.gen_dir, a.imgsz)
        Path(a.gen_dir, "model_info.json").write_text(json.dumps(res, indent=2))

In [ ]:
%%writefile /tmp/lm/src/aggregate.py
"""Collect run outputs (run.json + metrics.json) and build the result tables for the README/report.

Works on Kaggle (inputs = the training notebook versions) and locally on the downloaded outputs:
    python aggregate.py <dir-with-run-folders> [...] --out results/
"""

import argparse
import json
from pathlib import Path

import pandas as pd

METRICS = ["precision", "recall", "f1", "map50", "map50_95"]


def collect(roots, min_epochs=100):
    rows, seen = [], set()
    for root in roots:
        for run_json in sorted(Path(root).rglob("run.json")):
            run = json.loads(run_json.read_text())
            metrics_path = run_json.parent / "metrics.json"
            if run.get("status") != "done" or not metrics_path.exists() or run["run"] in seen:
                continue
            if run.get("epochs", 0) < min_epochs:  # smoke-test runs
                print(f"skipping {run_json.parent} ({run.get('epochs')} epochs)")
                continue
            seen.add(run["run"])
            m = json.loads(metrics_path.read_text())
            row = {"run": run["run"], "model": run["model"], "split": run["split"], "seed": run["seed"],
                   "params_M": m["params"] / 1e6, "gflops": m["gflops"], "train_hours": run.get("train_hours"),
                   "transferred": run.get("transferred_items")}
            for part in ("val", "test"):
                for k in METRICS:
                    row[f"{part}_{k}"] = m[part][k]
            rows.append(row)
    return pd.DataFrame(rows)


def summarise(df):
    """Mean and sample std over seeds for every (model, split)."""
    cols = [f"test_{k}" for k in METRICS] + [f"val_{k}" for k in METRICS]
    g = df.groupby(["split", "model"])
    out = g[cols].agg(["mean", "std"])
    out.columns = [f"{a}_{b}" for a, b in out.columns]
    out["n_seeds"] = g.size()
    out["params_M"] = g["params_M"].first()
    out["gflops"] = g["gflops"].first()
    return out.reset_index()


def paired_deltas(df):
    """CBAM minus baseline for the same (variant, split, seed) - the core ablation."""
    rows = []
    for (split, seed), grp in df.groupby(["split", "seed"]):
        by_model = grp.set_index("model")
        for base in ("yolov5s", "yolov5x"):
            cbam = f"{base}_cbam"
            if base in by_model.index and cbam in by_model.index:
                r = {"split": split, "seed": seed, "pair": f"{cbam} - {base}"}
                for k in METRICS:
                    r[f"d_test_{k}"] = by_model.loc[cbam, f"test_{k}"] - by_model.loc[base, f"test_{k}"]
                rows.append(r)
    return pd.DataFrame(rows)


def markdown_table(summary):
    lines = ["| Split | Model | Seeds | Params (M) | GFLOPs | P | R | mAP@0.5 | mAP@0.5:0.95 |",
             "|---|---|---|---|---|---|---|---|---|"]
    for _, r in summary.sort_values(["split", "model"]).iterrows():
        def f(k):
            s = r[f"test_{k}_std"]
            return f"{r[f'test_{k}_mean']:.3f}" + ("" if pd.isna(s) else f" ± {s:.3f}")
        lines.append(f"| {r['split']} | {r['model']} | {int(r['n_seeds'])} | {r['params_M']:.1f} | {r['gflops']:.1f} | "
                     f"{f('precision')} | {f('recall')} | {f('map50')} | {f('map50_95')} |")
    return "\n".join(lines)


def main(roots, out_dir, min_epochs=100):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    df = collect(roots, min_epochs)
    if df.empty:
        print("no finished runs found under", roots)
        return None
    df.sort_values(["split", "model", "seed"]).to_csv(out_dir / "runs.csv", index=False)
    summary = summarise(df)
    summary.to_csv(out_dir / "summary.csv", index=False)
    deltas = paired_deltas(df)
    deltas.to_csv(out_dir / "cbam_vs_baseline.csv", index=False)
    md = markdown_table(summary)
    (out_dir / "summary.md").write_text(md + "\n", encoding="utf-8")
    print(md)
    if not deltas.empty:
        print("\nCBAM minus baseline (test set), per seed:")
        print(deltas.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))
        agg = deltas.groupby(["split", "pair"])[[f"d_test_{k}" for k in METRICS]].agg(["mean", "std"])
        print("\nmean ± std of the difference:")
        print(agg.to_string(float_format=lambda x: f"{x:+.4f}"))
    return df, summary, deltas


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("roots", nargs="+")
    ap.add_argument("--out", default="results")
    ap.add_argument("--min-epochs", type=int, default=100, help="ignore shorter (smoke-test) runs")
    a = ap.parse_args()
    main(a.roots, a.out, a.min_epochs)

In [ ]:
%%writefile /tmp/lm/src/benchmark.py
"""Latency benchmark (forward pass only, batch 1, no NMS) for YOLOv5 and Ultralytics checkpoints.

Usage: python benchmark.py --yolo-dir /tmp/lm/yolov5 --weights a.pt b.pt --names a b --out latency.json
YOLOv5 checkpoints that contain CBAM need the CBAM-patched YOLOv5 checkout on the path.
"""

import argparse
import json
import os
import sys
import time
from pathlib import Path

ap = argparse.ArgumentParser()
ap.add_argument("--yolo-dir", required=True)
ap.add_argument("--weights", nargs="+", required=True)
ap.add_argument("--names", nargs="+", required=True)
ap.add_argument("--out", required=True)
ap.add_argument("--imgsz", type=int, default=416)
ap.add_argument("--gpu-iters", type=int, default=300)
ap.add_argument("--cpu-iters", type=int, default=30)
a = ap.parse_args()
weights = [str(Path(w).resolve()) for w in a.weights]
out_path = Path(a.out).resolve()

sys.path.insert(0, str(Path(a.yolo_dir).resolve()))
os.chdir(a.yolo_dir)
import torch  # noqa: E402
from ultralytics.utils.patches import torch_load  # noqa: E402


def load(path):
    ckpt = torch_load(path, map_location="cpu")
    model = (ckpt.get("ema") or ckpt["model"]).float()
    if hasattr(model, "fuse"):
        model = model.fuse()
    return model.eval()


@torch.no_grad()
def time_it(model, device, iters, half):
    model = model.to(device)
    x = torch.zeros(1, 3, a.imgsz, a.imgsz, device=device)
    if half:
        model, x = model.half(), x.half()
    for _ in range(max(iters // 10, 3)):
        model(x)
    if device == "cuda":
        torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(iters):
        model(x)
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.perf_counter() - t) / iters * 1000


res = {"imgsz": a.imgsz, "batch": 1, "note": "forward pass only, no NMS", "cpu_threads": torch.get_num_threads(),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "models": {}}
for name, w in zip(a.names, weights):
    model = load(w)
    r = {"weights": w, "cpu_fp32_ms": time_it(model, "cpu", a.cpu_iters, False)}
    if torch.cuda.is_available():
        r["gpu_fp16_ms"] = time_it(load(w), "cuda", a.gpu_iters, True)
        r["gpu_fp16_fps"] = 1000 / r["gpu_fp16_ms"]
    res["models"][name] = r
    print(name, {k: round(v, 2) for k, v in r.items() if k != "weights"})
out_path.write_text(json.dumps(res, indent=2))

In [ ]:
sys.path.insert(0, SRC_DIR)
sh(f"{sys.executable} {SRC_DIR}/setup_yolov5.py patch --yolo-dir {YOLO_DIR} --gen-dir {GEN_DIR}")  # to load CBAM models
import aggregate
res = aggregate.main(INPUT_ROOTS, OUT_DIR)
if res is None:
    raise SystemExit("No finished runs found: attach the training notebook versions as inputs.")
df, summary, deltas = res

In [ ]:
# ---- 2025 YOLO11m weights on the original test split (optional) ------------------------------
import splits
if LEGACY_YOLO11M:
    from ultralytics import YOLO
    ck = torch.load(LEGACY_YOLO11M, map_location="cpu", weights_only=False)
    legacy_imgsz = (ck.get("train_args") or {}).get("imgsz", 640)
    print("legacy YOLO11m was trained at imgsz", legacy_imgsz)
    src = splits.find_dataset(["/kaggle/input"])
    yamls = splits.build_splits(src, f"{WORK_DIR}/data")
    legacy = {}
    for split in ("val", "test"):
        r = YOLO(LEGACY_YOLO11M).val(data=str(yamls["random"]), split=split, imgsz=legacy_imgsz, batch=32,
                                     project=f"{OUT_DIR}/legacy_yolo11m", name=split, exist_ok=True)
        legacy[split] = {"precision": float(r.box.mp), "recall": float(r.box.mr), "map50": float(r.box.map50),
                         "map50_95": float(r.box.map)}
    legacy["imgsz"] = legacy_imgsz
    json.dump(legacy, open(f"{OUT_DIR}/legacy_yolo11m.json", "w"), indent=2)
    print(json.dumps(legacy, indent=2))

In [ ]:
# ---- latency: one checkpoint per architecture (seed 0 if available) ---------------------------
from pathlib import Path
picked = {}
for _, r in df.sort_values("seed").iterrows():
    if r["model"] not in picked:
        hits = list(Path("/kaggle/input").rglob(f"{r['run']}/weights/best.pt"))
        if hits:
            picked[r["model"]] = str(hits[0])
if LEGACY_YOLO11M:
    picked["yolo11m_legacy"] = LEGACY_YOLO11M
print(picked)
if picked:
    sh(f"{sys.executable} {SRC_DIR}/benchmark.py --yolo-dir {YOLO_DIR} --imgsz {IMGSZ} --out {OUT_DIR}/latency.json "
       f"--weights {' '.join(picked.values())} --names {' '.join(picked)}")

In [ ]:
# ---- plots -----------------------------------------------------------------------------------
import matplotlib.pyplot as plt
for split, grp in df.groupby("split"):
    order = [m for m in ("yolov5s", "yolov5s_cbam", "yolov5x", "yolov5x_cbam", "yolo11m") if m in set(grp["model"])]
    fig, ax = plt.subplots(figsize=(7, 4))
    for i, m in enumerate(order):
        v = grp[grp["model"] == m]["test_map50_95"]
        ax.bar(i, v.mean(), yerr=v.std() if len(v) > 1 else None, capsize=4, color="#9ab" if "cbam" not in m else "#e96")
        ax.scatter([i] * len(v), v, color="k", s=12, zorder=3)
    ax.set_xticks(range(len(order)), order, rotation=20)
    ax.set_ylabel("test mAP@0.5:0.95")
    ax.set_title(f"{split} split (bars: mean, dots: seeds)")
    lo = grp["test_map50_95"].min()
    ax.set_ylim(max(0, lo - 0.08), min(1, grp["test_map50_95"].max() + 0.04))
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/map50_95_{split}.png", dpi=150)
    plt.show()